In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import DBSCAN

print("Student B environment is working!")

In [ ]:
import os

print(os.listdir("../data/raw"))

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/processed_data.csv")

print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import folium

from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("All libraries imported successfully.")

In [ ]:
df = pd.read_csv("../data/raw/processed_data.csv")

In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0]

print("Missing Columns :", len(missing))

missing.sort_values(ascending=False)

In [ ]:
df.drop(columns=["Year", "Month", "Day"], inplace=True)

df["Hour"] = df["Hour"].fillna(df["Hour"].mode()[0])

In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df.describe().T

In [ ]:
invalid_lat = (df["Latitude"]==0).sum()

invalid_lon = (df["Longitude"]==0).sum()

print("Invalid Latitude :",invalid_lat)

print("Invalid Longitude :",invalid_lon)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x="Accident_Severity",
    data=df,
    order=df["Accident_Severity"].value_counts().index
)

plt.title("Accident Severity Distribution")
plt.xlabel("Severity")
plt.ylabel("Count")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="Speed_limit",
    y="Number_of_Casualties"
)

plt.title("Speed Limit vs Number of Casualties")
plt.show()

In [ ]:
numerical = [

"Latitude",

"Longitude",

"Speed_limit",

"Number_of_Casualties",

"Number_of_Vehicles",

"Road_Risk_Score"

]

plt.figure(figsize=(10,8))

sns.heatmap(

df[numerical].corr(),

annot=True,

cmap="coolwarm"

)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

plt.scatter(

df["Longitude"],

df["Latitude"],

s=1,

alpha=0.3

)

plt.title("Accident Locations")

plt.xlabel("Longitude")

plt.ylabel("Latitude")

plt.show()

In [ ]:
memory = df.memory_usage(deep=True).sum()/1024**2

print(f"Dataset Memory Usage : {memory:.2f} MB")

In [ ]:
print("="*60)
print("EDA SUMMARY")
print("="*60)

print(f"Rows               : {df.shape[0]}")
print(f"Columns            : {df.shape[1]}")
print(f"Missing Values     : {df.isnull().sum().sum()}")
print(f"Duplicate Records  : {df.duplicated().sum()}")
print(f"Memory Usage (MB)  : {memory:.2f}")

print("="*60)

In [ ]:
# ============================================================
# Remove Unnecessary Columns
# ============================================================

columns_to_drop = ["Year", "Month", "Day"]

existing_columns = [col for col in columns_to_drop if col in df.columns]

df.drop(columns=existing_columns, inplace=True)

print("Removed Columns:", existing_columns)

In [ ]:
# ============================================================
# Fill Missing Hour Values
# ============================================================

if "Hour" in df.columns:
    df["Hour"].fillna(df["Hour"].mode()[0], inplace=True)

print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
def reduce_memory(df):

    start = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:

        col_type = df[col].dtype

        if pd.api.types.is_integer_dtype(col_type):

            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)

            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)

            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(col_type):

            df[col] = df[col].astype(np.float32)

    end = df.memory_usage(deep=True).sum() / 1024**2

    print(f"Memory Before : {start:.2f} MB")
    print(f"Memory After  : {end:.2f} MB")

    return df

df = reduce_memory(df)

In [ ]:
geo_df = df[["Latitude", "Longitude"]].copy()

geo_df = geo_df.dropna()

geo_df = geo_df[
    (geo_df["Latitude"] != 0) &
    (geo_df["Longitude"] != 0)
]

print("Valid Coordinates :", len(geo_df))
print(geo_df.describe())

In [ ]:
plt.figure(figsize=(10, 8))

plt.scatter(
    geo_df["Longitude"],
    geo_df["Latitude"],
    s=0.5,
    alpha=0.2
)

plt.title("Accident Locations Across UK")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

sns.histplot(geo_df["Latitude"], bins=50)

plt.title("Latitude Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

sns.histplot(geo_df["Longitude"], bins=50)

plt.title("Longitude Distribution")

plt.show()

In [ ]:
scaler = StandardScaler()

geo_scaled = scaler.fit_transform(geo_df)

print("Scaled Shape :", geo_scaled.shape)

geo_df.to_csv(
    "../data/output/processed_coordinates.csv",
    index=False
)

print("Coordinates Saved Successfully.")

In [ ]:
print("=" * 50)

print("PREPROCESSING SUMMARY")

print("=" * 50)

print("Original Dataset :", df.shape)

print("Coordinates Used :", geo_df.shape)

print("Missing Values :", geo_df.isnull().sum().sum())

print("=" * 50)

In [ ]:
coords = np.radians(
    geo_df[["Latitude", "Longitude"]].values
)

print(coords.shape)

In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
EARTH_RADIUS = 6371000

eps = 500 / EARTH_RADIUS

min_samples = 25

print("eps:", eps)

In [ ]:
dbscan = DBSCAN(
    eps=eps,
    min_samples=min_samples,
    metric="haversine",
    algorithm="ball_tree"
)

clusters = dbscan.fit_predict(coords)

geo_df["Cluster"] = clusters

In [ ]:
print("Number of Clusters")

print(geo_df["Cluster"].nunique() - 1)

print()

print("Noise Points")

print((geo_df["Cluster"] == -1).sum())

In [ ]:
cluster_counts = geo_df["Cluster"].value_counts()

cluster_counts.head(20)

In [ ]:
hotspots = geo_df[geo_df["Cluster"] != -1]

print(hotspots.shape)

In [ ]:
sample = hotspots.sample(
    min(50000, len(hotspots)),
    random_state=42
)


In [ ]:
plt.figure(figsize=(12, 9))

plt.scatter(
    sample["Longitude"],
    sample["Latitude"],
    c=sample["Cluster"],
    s=2,
    cmap="tab20",
    alpha=0.6
)

plt.title("DBSCAN Hotspots")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
cluster_size = hotspots["Cluster"].value_counts()

cluster_size.head(15)

In [ ]:
largest = cluster_size.reset_index()

largest.columns = ["Cluster", "Accidents"]

largest.head(10)


In [ ]:
cluster_centers = hotspots.groupby("Cluster").agg({
    "Latitude": "mean",
    "Longitude": "mean"
}).reset_index()

cluster_centers.head()

In [ ]:
geo_df.to_csv(
    "../data/output/accident_clusters.csv",
    index=False
)

In [ ]:
print("=" * 60)

print("DBSCAN SUMMARY")

print("=" * 60)

print("Clusters Found :", cluster_centers.shape[0])

print("Noise Points :", len(geo_df) - len(hotspots))

print("Largest Cluster :", cluster_size.max())

print("=" * 60)

In [ ]:
df_clustered = df.copy()

df_clustered["Cluster"] = -1

df_clustered.loc[geo_df.index, "Cluster"] = geo_df["Cluster"]

print(df_clustered.shape)

In [ ]:
hotspot_df = df_clustered[df_clustered["Cluster"] != -1].copy()

print(hotspot_df.shape)

In [ ]:
cluster_centers = hotspot_df.groupby("Cluster").agg({
    "Latitude": "mean",
    "Longitude": "mean"
}).reset_index()

cluster_centers.rename(columns={
    "Latitude": "Center_Latitude",
    "Longitude": "Center_Longitude"
}, inplace=True)

cluster_centers.head()

In [ ]:
accident_count = hotspot_df.groupby("Cluster").size().reset_index(name="Total_Accidents")

accident_count.head()

In [ ]:
severity = hotspot_df.groupby("Cluster")["Accident_Severity"] \
    .agg(lambda x: x.mode().iloc[0]) \
    .reset_index()

severity.rename(columns={
    "Accident_Severity": "Dominant_Severity"
}, inplace=True)

severity.head()

In [ ]:
weather_cols = [
    col for col in hotspot_df.columns
    if col.startswith("Weather_Conditions_")
]

weather_cols[:5]

In [ ]:
weather_summary = hotspot_df.groupby("Cluster")[weather_cols].sum()

dominant_weather = weather_summary.idxmax(axis=1)

dominant_weather = dominant_weather.str.replace(
    "Weather_Conditions_",
    "",
    regex=False
)

dominant_weather = dominant_weather.reset_index()

dominant_weather.columns = [
    "Cluster",
    "Dominant_Weather"
]

dominant_weather.head()


In [ ]:
road_cols = [
    col for col in hotspot_df.columns
    if col.startswith("Road_Type_")
]

road_summary = hotspot_df.groupby("Cluster")[road_cols].sum()

dominant_road = road_summary.idxmax(axis=1)

dominant_road = dominant_road.str.replace(
    "Road_Type_",
    "",
    regex=False
)

dominant_road = dominant_road.reset_index()

dominant_road.columns = [
    "Cluster",
    "Dominant_Road_Type"
]

In [ ]:
speed = hotspot_df.groupby("Cluster")["Speed_limit"] \
    .mean() \
    .round(2) \
    .reset_index()

speed.rename(columns={
    "Speed_limit": "Average_Speed"
}, inplace=True)

In [ ]:
casualties = hotspot_df.groupby("Cluster")["Number_of_Casualties"] \
    .mean() \
    .round(2) \
    .reset_index()

casualties.rename(columns={
    "Number_of_Casualties": "Average_Casualties"
}, inplace=True)

In [ ]:
hour = hotspot_df.groupby("Cluster")["Hour"] \
    .agg(lambda x: x.mode().iloc[0]) \
    .reset_index()

hour.rename(columns={
    "Hour": "Peak_Hour"
}, inplace=True)

In [ ]:
hotspot_summary = cluster_centers

hotspot_summary = hotspot_summary.merge(
    accident_count,
    on="Cluster"
)

hotspot_summary = hotspot_summary.merge(
    severity,
    on="Cluster"
)

hotspot_summary = hotspot_summary.merge(
    dominant_weather,
    on="Cluster"
)

hotspot_summary = hotspot_summary.merge(
    dominant_road,
    on="Cluster"
)

hotspot_summary = hotspot_summary.merge(
    speed,
    on="Cluster"
)

hotspot_summary = hotspot_summary.merge(
    casualties,
    on="Cluster"
)

hotspot_summary = hotspot_summary.merge(
    hour,
    on="Cluster"
)

hotspot_summary.head()

In [ ]:
hotspot_summary = hotspot_summary.sort_values(
    "Total_Accidents",
    ascending=False
)

hotspot_summary.reset_index(drop=True, inplace=True)

hotspot_summary.head(20)

In [ ]:
hotspot_summary.to_csv(
    "../data/output/hotspot_summary.csv",
    index=False
)

print("Hotspot Summary Saved Successfully.")

In [ ]:
print("=" * 60)

print("HOTSPOT PROFILE SUMMARY")

print("=" * 60)

print("Total Hotspots :", len(hotspot_summary))

print()

print("Largest Hotspot")

print(hotspot_summary.iloc[0])

print("=" * 60)

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
sample_size = min(20000, len(geo_df))

sample_geo = geo_df.sample(
    n=sample_size,
    random_state=42
)

coords_sample = np.radians(
    sample_geo[["Latitude", "Longitude"]].values
)

print("Sample Size:", len(coords_sample))

In [ ]:
k = 25

neighbors = NearestNeighbors(
    n_neighbors=k,
    metric="haversine",
    algorithm="ball_tree"
)

neighbors.fit(coords_sample)

In [ ]:
distances, indices = neighbors.kneighbors(coords_sample)

k_distance = np.sort(distances[:, k - 1])

In [ ]:
EARTH_RADIUS = 6371000

k_distance_meters = k_distance * EARTH_RADIUS

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(k_distance_meters)

plt.xlabel("Sorted Points")
plt.ylabel("Distance to 25th Neighbor (meters)")
plt.title("K-Distance Graph for DBSCAN")

plt.grid(True)

plt.show()

In [ ]:
eps = 500 / EARTH_RADIUS

In [ ]:
print("=" * 50)
print("Suggested DBSCAN Parameters")
print("=" * 50)
print("min_samples = 25")
print("Choose eps from the elbow of the graph")
print("=" * 50)

In [ ]:
from folium.plugins import HeatMap, MarkerCluster

In [ ]:
uk_map = folium.Map(
    location=[
        hotspot_summary["Center_Latitude"].mean(),
        hotspot_summary["Center_Longitude"].mean()
    ],
    zoom_start=6,
    tiles="OpenStreetMap"
)

print("Base Map Created")

In [ ]:
marker_cluster = MarkerCluster(
    name="Hotspot Clusters"
)

uk_map.add_child(marker_cluster)

In [ ]:
# ============================================================
# HeatMap Layer
# ============================================================

from folium.plugins import HeatMap

heat_data = hotspot_df[["Latitude", "Longitude"]].values.tolist()

HeatMap(
    heat_data,
    radius=10,
    blur=15,
    name="Accident Density"
).add_to(uk_map)

print("HeatMap Layer Added")

In [ ]:
# ============================================================
# Color Function
# ============================================================

def hotspot_color(accidents):

    if accidents >= 2000:
        return "darkred"

    elif accidents >= 1000:
        return "red"

    elif accidents >= 500:
        return "orange"

    elif accidents >= 200:
        return "blue"

    else:
        return "green"

In [ ]:
# ============================================================
# Add Hotspot Markers
# ============================================================

for _, row in hotspot_summary.iterrows():

    popup = f"""
    <b>Hotspot #{int(row.Cluster)}</b><br><br>

    🚗 <b>Total Accidents:</b> {int(row.Total_Accidents)}<br>

    ⚠ <b>Severity:</b> {row.Dominant_Severity}<br>

    🌧 <b>Weather:</b> {row.Dominant_Weather}<br>

    🛣 <b>Road Type:</b> {row.Dominant_Road_Type}<br>

    🚑 <b>Average Casualties:</b> {row.Average_Casualties}<br>

    🚦 <b>Average Speed:</b> {row.Average_Speed} km/h<br>

    🕒 <b>Peak Hour:</b> {int(row.Peak_Hour)}
    """

    folium.CircleMarker(
        location=[
            row.Center_Latitude,
            row.Center_Longitude
        ],
        radius=max(5, min(row.Total_Accidents / 100, 20)),
        color=hotspot_color(row.Total_Accidents),
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup, max_width=350)
    ).add_to(marker_cluster)

uk_map.save("../data/output/uk_hotspot_map.html")

print("Hotspot markers added successfully!")
print("Map saved to: data/output/uk_hotspot_map.html")

In [ ]:
# ============================================================
# Layer Control
# ============================================================

folium.LayerControl().add_to(uk_map)

uk_map.save("../data/output/uk_hotspot_map.html")

print("Layer Control Added")
print("Final map saved to: data/output/uk_hotspot_map.html")


In [ ]:
uk_map


In [ ]:
largest_hotspot = hotspot_summary.iloc[0]

result = {
    "cluster": int(largest_hotspot["Cluster"]),
    "hotspot": True,
    "accidents": int(largest_hotspot["Total_Accidents"])
}

print(result)